# Tempo sentado e saúde mental em escolares: agrupamento e regras de associação
**Disciplina:** Mineração de Dados · **Base:** PeNSE 2024 (IBGE)

Notebook com todo o código do projeto, na mesma ordem do relatório:
1. Base de dados e pré-processamento
2. Análise exploratória (tempo sentado × saúde mental)
3. Agrupamento (K-Means)
4. Regras de associação (Apriori, implementado do zero)

> `B03010C` mede o tempo sentado por dia (TV, videogame, computador, celular, tablet etc.), sem contar fim de semana, feriados e tempo na escola. Não isola telas: é uma aproximação do tempo de tela e do sedentarismo.

In [ ]:
import os
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score, adjusted_rand_score)

CAMINHO = "PENSE2024_MICRODADOS.csv"   # ajuste se o CSV estiver em outra pasta
SEMENTE = 42

os.makedirs("figuras", exist_ok=True)
os.makedirs("resultados", exist_ok=True)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## 1. Base de dados e pré-processamento
Microdados da Pesquisa Nacional de Saúde do Escolar (PeNSE) 2024, do IBGE. O arquivo tem 430 colunas codificadas; usamos apenas as 11 abaixo.

| Coluna | Significado | Códigos |
|---|---|---|
| `PESO_ALUNO_FREQ` | peso amostral | numérico |
| `B01001A` | sexo | 1 homem, 2 mulher |
| `IDADE_AGREG` | faixa etária | 1 menos de 13, 2 de 13 a 15, 3 de 16 a 17, 4 de 18 ou mais |
| `B03010C` | tempo sentado por dia | 1 (até 1 h) a 9 (mais de 8 h); 10 não sei |
| `B12003` | amigos próximos | 1 nenhum, 2 um, 3 dois, 4 três ou mais |
| `B12004` | preocupação | 1 nunca ... 5 sempre |
| `B12005` | tristeza | 1 nunca ... 5 sempre |
| `B12006` | sente que ninguém se preocupa com ele | 1 nunca ... 5 sempre |
| `B12007` | irritação | 1 nunca ... 5 sempre |
| `B12008` | sente que a vida não vale a pena | 1 nunca ... 5 sempre |
| `B12009` | vontade de se machucar (12 meses) | 1 sim, 2 não |

In [ ]:
colunas = ["PESO_ALUNO_FREQ", "B01001A", "IDADE_AGREG", "B03010C",
           "B12003", "B12004", "B12005", "B12006", "B12007", "B12008", "B12009"]
respostas = [c for c in colunas if c != "PESO_ALUNO_FREQ"]

bruto = pd.read_csv(CAMINHO, sep=",", encoding="utf-8", usecols=colunas)
print("Registros no arquivo:", len(bruto))

# 1) linhas sem questionário respondido (todas as respostas vazias)
df = bruto.dropna(subset=["B01001A"]).reset_index(drop=True)
print("Com questionário respondido:", len(df))

# 2) códigos negativos (-9, -2...) = sem resposta/inválido -> ausente
df[respostas] = df[respostas].mask(df[respostas] < 0)

# 3) tempo sentado: código 10 = "não sei" -> ausente
df["B03010C"] = df["B03010C"].where(df["B03010C"] <= 9)

# 4) população-alvo da PeNSE: 13 a 17 anos (faixas 2 e 3)
df = df[df["IDADE_AGREG"].isin([2, 3])].reset_index(drop=True)
print("Entre 13 e 17 anos:", len(df))

# 5) registros completos nas variáveis do estudo (exclusão por linha)
modelo = ["B03010C", "B12003", "B12004", "B12005", "B12006", "B12007", "B12008", "B12009"]
print("Valores ausentes por variável:\n", df[modelo].isna().sum().to_dict())
analise = df.dropna(subset=modelo).reset_index(drop=True)
print("Registros completos usados no estudo:", len(analise),
      f"({100*len(analise)/len(df):.1f}% dos de 13 a 17 anos)")

**Justificativas do pré-processamento**
- Linhas vazias e códigos negativos não são respostas reais; mantê-los distorceria médias e distâncias.
- Restringimos a 13 a 17 anos porque é a população-alvo da PeNSE, o que permite comparar com os resultados publicados pelo IBGE.
- Excluímos os registros incompletos em vez de imputar: a maior parte da perda é o "não sei" do tempo sentado, e inventar um valor para ele mascararia justamente a variável central do estudo.
- Os indicadores binários usam a regra "na maioria das vezes ou sempre" (códigos 4 e 5), a mesma lógica dos indicadores do IBGE.

In [ ]:
def grupo_tempo(v):
    if v <= 2: return "até 2 h"
    if v <= 4: return "2 a 4 h"
    if v <= 6: return "4 a 6 h"
    return "mais de 6 h"

ORDEM = ["até 2 h", "2 a 4 h", "4 a 6 h", "mais de 6 h"]

d = analise.copy()
d["tempo_sentado"] = pd.Categorical(d["B03010C"].map(grupo_tempo), categories=ORDEM, ordered=True)
d["sexo"] = d["B01001A"].map({1: "homem", 2: "mulher"})

d["preocupado"]            = d["B12004"] >= 4
d["triste"]                = d["B12005"] >= 4
d["ninguem_se_preocupa"]   = d["B12006"] >= 4
d["irritado"]              = d["B12007"] >= 4
d["vida_nao_vale_a_pena"]  = d["B12008"] >= 4
d["vontade_se_machucar"]   = d["B12009"] == 1
d["nenhum_amigo_proximo"]  = d["B12003"] == 1

INDICADORES = ["preocupado", "triste", "ninguem_se_preocupa", "irritado",
               "vida_nao_vale_a_pena", "vontade_se_machucar", "nenhum_amigo_proximo"]

d["tempo_sentado"].value_counts().reindex(ORDEM)

## 2. Análise exploratória: tempo sentado × saúde mental
Porcentagens ponderadas pelo peso amostral (`PESO_ALUNO_FREQ`), para valer para os escolares do país e não só para quem respondeu.

In [ ]:
def pct_ponderado(dados, grupo, indicadores):
    linhas = {}
    for g, x in dados.groupby(grupo, observed=True):
        linha = {i: 100 * x.loc[x[i], "PESO_ALUNO_FREQ"].sum() / x["PESO_ALUNO_FREQ"].sum()
                 for i in indicadores}
        linha["n"] = len(x)
        linhas[g] = linha
    return pd.DataFrame(linhas).T

# Validação: prevalências gerais (13-17 anos, ponderadas) para comparar com a divulgação do IBGE
geral = {i: 100 * d.loc[d[i], "PESO_ALUNO_FREQ"].sum() / d["PESO_ALUNO_FREQ"].sum() for i in INDICADORES}
print("Prevalência geral ponderada (%):")
print(pd.Series(geral).round(1))

In [ ]:
tab = pct_ponderado(d, "tempo_sentado", INDICADORES)
tab.round(1).to_csv("resultados/exploratoria_tempo_sentado.csv")
tab.round(1)

In [ ]:
(tab.loc["mais de 6 h", INDICADORES] - tab.loc["até 2 h", INDICADORES]).round(1).sort_values(ascending=False)

In [ ]:
ax = tab[INDICADORES].plot(kind="bar", figsize=(11, 5), width=0.85)
ax.set_title("Indicadores de saúde mental por tempo sentado por dia (PeNSE 2024, ponderado)")
ax.set_xlabel("Tempo sentado por dia"); ax.set_ylabel("% dos alunos")
ax.set_xticklabels(ORDEM, rotation=0)
ax.legend(title="Indicador", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout(); plt.savefig("figuras/01_exploratoria.png", dpi=150); plt.show()

Sexo costuma estar ligado tanto ao tempo de tela quanto à saúde mental; por isso repetimos a tabela separando por sexo.

In [ ]:
for s in ["homem", "mulher"]:
    print(f"--- {s} ---")
    print(pct_ponderado(d[d["sexo"] == s], "tempo_sentado", INDICADORES).round(1))
    print()

**Tamanho da associação.** As porcentagens mostram diferenças entre faixas; a correlação de Spearman entre o tempo sentado (1 a 9) e um escore geral de sofrimento (soma dos 7 indicadores padronizados, com valores maiores = mais sofrimento) mostra o quão forte é a relação no conjunto.

In [ ]:
from scipy.stats import spearmanr

escala = analise[["B12004", "B12005", "B12006", "B12007", "B12008"]].copy()
escala["vontade_se_machucar"] = (analise["B12009"] == 1).astype(int)
escala["poucos_amigos"] = 4 - analise["B12003"]
escore = ((escala - escala.mean()) / escala.std()).sum(axis=1)

rho = spearmanr(analise["B03010C"], escore)[0]
print(f"Spearman (tempo sentado x escore geral): {rho:.3f}  (rho ao quadrado = {rho**2:.3f})")
for s, cod in [("homem", 1), ("mulher", 2)]:
    m = analise["B01001A"] == cod
    print(f"  {s}: {spearmanr(analise.loc[m, 'B03010C'], escore[m])[0]:.3f}")
print("Por variável:")
for c in escala.columns:
    print(f"  {c}: {spearmanr(analise['B03010C'], escala[c])[0]:.3f}")

## 3. Agrupamento (K-Means)
**Objetivo:** identificar perfis de escolares a partir do tempo sentado e de oito medidas de saúde mental, e verificar se o tempo sentado se distribui de forma diferente entre os perfis.

**Pré-processamento específico:** as variáveis entram na escala original (ordinais de 1 a 5, binária 0/1) e são padronizadas (`StandardScaler`), pois o K-Means usa distância euclidiana e escalas diferentes dariam pesos diferentes. A variável `poucos_amigos` (= 4 - `B12003`) é invertida para que, em todas as variáveis, valores maiores signifiquem mais problema. Sexo e idade **não** entram no modelo; são usados depois para descrever os grupos.

In [ ]:
X = pd.DataFrame({
    "tempo_sentado":       analise["B03010C"],
    "preocupado":          analise["B12004"],
    "triste":              analise["B12005"],
    "ninguem_se_preocupa": analise["B12006"],
    "irritado":            analise["B12007"],
    "vida_nao_vale":       analise["B12008"],
    "vontade_se_machucar": (analise["B12009"] == 1).astype(int),
    "poucos_amigos":       4 - analise["B12003"],
})
Z = StandardScaler().fit_transform(X)
print(X.shape)
X.describe().loc[["mean", "std", "min", "max"]].round(2)

**Número de grupos.** Testamos K de 2 a 8 e comparamos: inércia (método do cotovelo), coeficiente de silhueta (calculado em amostra de 10 mil registros por custo computacional), índice de Davies-Bouldin (menor é melhor) e Calinski-Harabasz (maior é melhor). Como são dados de questionário, sem separação natural forte, também consideramos a interpretabilidade dos grupos.

In [ ]:
amostra = np.random.RandomState(SEMENTE).choice(len(Z), 10000, replace=False)
linhas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEMENTE).fit(Z)
    linhas.append({
        "k": k,
        "inercia": km.inertia_,
        "silhueta": silhouette_score(Z[amostra], km.labels_[amostra]),
        "davies_bouldin": davies_bouldin_score(Z, km.labels_),
        "calinski_harabasz": calinski_harabasz_score(Z, km.labels_),
    })
selecao = pd.DataFrame(linhas).set_index("k")
selecao.round(3).to_csv("resultados/selecao_k.csv")
selecao.round(3)

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(14, 4))
selecao["inercia"].plot(ax=eixos[0], marker="o", title="Cotovelo (inércia)")
selecao["silhueta"].plot(ax=eixos[1], marker="o", title="Silhueta (maior é melhor)")
selecao["davies_bouldin"].plot(ax=eixos[2], marker="o", title="Davies-Bouldin (menor é melhor)")
for e in eixos: e.set_xlabel("número de grupos (k)")
plt.tight_layout(); plt.savefig("figuras/02_selecao_k.png", dpi=150); plt.show()

**Escolha de K = 4.** Não há cotovelo nítido (a inércia cai de forma gradual) e a silhueta é baixa para K >= 3 (entre 0,14 e 0,17), o que indica estrutura fraca. K = 2 tem a melhor silhueta, mas apenas separa "mais sofrimento" de "menos sofrimento". Com K = 5 o grupo extra nasce da divisão do grupo intermediário pela variável binária `vontade_se_machucar` (todos 1 de um lado, todos 0 do outro), sem perfil novo. K = 4 mantém um perfil de baixo sofrimento, um intermediário, um de sofrimento elevado e um perfil distinto de **poucos amigos**, e é muito estável entre sementes (índice de Rand ajustado acima de 0,98). A escolha, portanto, combina métricas, estabilidade e interpretabilidade.

In [ ]:
K = 4   # justificativa na célula anterior: interpretabilidade e estabilidade, já que as métricas isoladas não indicam um K claro
modelo_km = KMeans(n_clusters=K, n_init=10, random_state=SEMENTE).fit(Z)
analise["grupo"] = modelo_km.labels_

# Estabilidade: o mesmo agrupamento com outras sementes (ARI = 1 significa partições idênticas)
ari = [adjusted_rand_score(modelo_km.labels_,
       KMeans(n_clusters=K, n_init=10, random_state=s).fit(Z).labels_) for s in [1, 7, 21, 99]]
print("Índice de Rand ajustado entre sementes:", np.round(ari, 3))
print("Silhueta (amostra):", round(silhouette_score(Z[amostra], modelo_km.labels_[amostra]), 3))
print("Davies-Bouldin:", round(davies_bouldin_score(Z, modelo_km.labels_), 3))

In [ ]:
perfil = X.copy()
perfil["grupo"] = analise["grupo"]
perfil["mulher_%"] = 100 * (analise["B01001A"] == 2)
perfil["mais_de_6h_%"] = 100 * (analise["B03010C"] >= 7)
resumo = perfil.groupby("grupo").mean()
resumo["n"] = perfil.groupby("grupo").size()
resumo["%_da_amostra"] = 100 * resumo["n"] / resumo["n"].sum()

# nomes automáticos (não dependem da numeração aleatória dos grupos)
nomes = {}
iso = resumo["poucos_amigos"].idxmax();            nomes[iso] = "Poucos amigos"
resto = resumo.drop(iso)
alto = resto["vida_nao_vale"].idxmax();            nomes[alto] = "Sofrimento elevado"
baixo = resto.drop(alto)["triste"].idxmin();       nomes[baixo] = "Baixo sofrimento"
for g in resumo.index: nomes.setdefault(g, "Preocupados e irritados")

resumo.insert(0, "perfil", resumo.index.map(nomes))
resumo = resumo.sort_values("tempo_sentado")
resumo.round(2).to_csv("resultados/perfil_grupos.csv")
resumo.round(2)

In [ ]:
colunas_perfil = ["tempo_sentado", "preocupado", "triste", "ninguem_se_preocupa", "irritado",
                  "vida_nao_vale", "vontade_se_machucar", "poucos_amigos"]
padr = (resumo[colunas_perfil] - X.mean()) / X.std()      # médias dos grupos em desvios-padrão

fig, ax = plt.subplots(figsize=(10, 3.8))
im = ax.imshow(padr.values, cmap="RdBu_r", vmin=-1.2, vmax=1.2, aspect="auto")
ax.set_xticks(range(len(colunas_perfil))); ax.set_xticklabels(colunas_perfil, rotation=30, ha="right")
ax.set_yticks(range(len(resumo))); ax.set_yticklabels(resumo["perfil"])
for i in range(padr.shape[0]):
    for j in range(padr.shape[1]):
        ax.text(j, i, f"{padr.values[i, j]:.1f}", ha="center", va="center", fontsize=8,
                color="white" if abs(padr.values[i, j]) > 0.8 else "black")
ax.set_title("Perfil dos grupos (média em desvios-padrão da amostra)")
plt.colorbar(im, ax=ax); plt.tight_layout()
plt.savefig("figuras/03_perfil_grupos.png", dpi=150); plt.show()

In [ ]:
# Visualização em 2 dimensões (PCA) de uma amostra
pca = PCA(n_components=2, random_state=SEMENTE).fit(Z)
P = pca.transform(Z[amostra])
print("Variância explicada pelas 2 componentes:", pca.explained_variance_ratio_.round(3))

plt.figure(figsize=(7, 5))
for g in resumo.index:
    m = analise["grupo"].values[amostra] == g
    plt.scatter(P[m, 0], P[m, 1], s=5, alpha=0.4, label=nomes[g])
plt.legend(markerscale=3); plt.xlabel("componente 1"); plt.ylabel("componente 2")
plt.title("Grupos projetados em 2 componentes (PCA)")
plt.tight_layout(); plt.savefig("figuras/04_pca_grupos.png", dpi=150); plt.show()

## 4. Regras de associação (Apriori)
**Objetivo:** encontrar combinações frequentes entre faixa de tempo sentado, sexo e indicadores de saúde mental, e medir a força de cada regra.

**Pré-processamento específico:** cada aluno vira uma "transação" com itens verdadeiro/falso: uma faixa de tempo sentado (4 itens), sexo (2 itens) e os 7 indicadores de saúde mental (presente = o aluno apresenta o indicador). Não usamos itens para a ausência dos indicadores.

**Métricas:** *suporte* (fração dos alunos que têm o conjunto de itens), *confiança* (entre os alunos que têm o antecedente, fração que também tem o consequente) e *lift* (confiança dividida pela frequência do consequente; acima de 1 indica associação positiva). Também calculamos *leverage*.

O algoritmo Apriori foi implementado do zero (geração de candidatos por nível e poda por suporte mínimo, conforme Agrawal e Srikant, 1994), para não depender de bibliotecas extras.

In [ ]:
T = pd.DataFrame({
    "sexo: mulher": d["sexo"] == "mulher",
    "sexo: homem":  d["sexo"] == "homem",
})
for g in ORDEM:
    T[f"tempo: {g}"] = d["tempo_sentado"] == g
for i in INDICADORES:
    T[i.replace("_", " ")] = d[i]

def apriori(B, min_support, max_len=4):
    """Retorna {frozenset(itens): suporte} para todos os conjuntos frequentes."""
    M, nomes_itens = B.values, list(B.columns)
    atual = {(i,): M[:, i].mean() for i in range(len(nomes_itens))}
    atual = {k: v for k, v in atual.items() if v >= min_support}
    todos, tam = dict(atual), 1
    while atual and tam < max_len:
        candidatos = set()
        chaves = sorted(atual)
        for a, b in combinations(chaves, 2):
            if a[:-1] == b[:-1]:                       # junta conjuntos que diferem só no último item
                c = a + (b[-1],)
                if all(s in atual for s in combinations(c, tam)):   # poda: todos os subconjuntos frequentes
                    candidatos.add(c)
        novo = {}
        for c in candidatos:
            s = M[:, list(c)].all(axis=1).mean()
            if s >= min_support:
                novo[c] = s
        todos.update(novo); atual = novo; tam += 1
    return {frozenset(nomes_itens[i] for i in k): v for k, v in todos.items()}

MIN_SUPORTE = 0.005   # baixo o bastante para incluir também o indicador raro (nenhum amigo próximo)
freq = apriori(T, MIN_SUPORTE)
print("Conjuntos frequentes:", len(freq))

In [ ]:
def gerar_regras(freq):
    linhas = []
    for conjunto, sup in freq.items():
        if len(conjunto) < 2: continue
        for cons in conjunto:                          # um único item no consequente
            ant = conjunto - {cons}
            conf = sup / freq[ant]
            sup_c = freq[frozenset([cons])]
            linhas.append({
                "antecedente": " + ".join(sorted(ant)), "consequente": cons,
                "suporte": sup, "confianca": conf, "lift": conf / sup_c,
                "leverage": sup - freq[ant] * sup_c,
            })
    return pd.DataFrame(linhas)

regras = gerar_regras(freq)
print("Regras geradas:", len(regras))

MIN_CONFIANCA, MIN_LIFT, MIN_SUPORTE_REGRA = 0.30, 1.10, 0.03
filtradas = regras[(regras.confianca >= MIN_CONFIANCA) & (regras.lift >= MIN_LIFT)
                   & (regras.suporte >= MIN_SUPORTE_REGRA)]
print("Regras com confiança >= 30%, lift >= 1,1 e suporte >= 3%:", len(filtradas))

In [ ]:
nomes_ind = [i.replace("_", " ") for i in INDICADORES]

# (a) Regras simples: uma faixa de tempo sentado -> um indicador
simples = regras[regras.antecedente.str.fullmatch("tempo: .+") & regras.consequente.isin(nomes_ind)]
print("Lift (linhas = faixa de tempo sentado; colunas = indicador):")
print(simples.pivot(index="antecedente", columns="consequente", values="lift")
             .reindex([f"tempo: {g}" for g in ORDEM]).round(2))
print()
print("Confiança (%):")
print((100 * simples.pivot(index="antecedente", columns="consequente", values="confianca"))
             .reindex([f"tempo: {g}" for g in ORDEM]).round(1))

In [ ]:
# (b) Principais regras com tempo sentado > 6 h no antecedente, ordenadas por lift
principais = (filtradas[filtradas.antecedente.str.contains("tempo: mais de 6 h")
                        & filtradas.consequente.isin(nomes_ind)]
              .sort_values("lift", ascending=False).head(10))
principais.round(3).to_csv("resultados/regras_tempo_mais_de_6h.csv", index=False)
principais.round(3)

In [ ]:
# (c) Regras para sexo: o quanto o sexo explica os indicadores?
sexo = regras[regras.antecedente.eq("sexo: mulher") & regras.consequente.isin(nomes_ind)].sort_values("lift", ascending=False)
sexo.round(3)

In [ ]:
regras.sort_values("lift", ascending=False).round(4).to_csv("resultados/todas_as_regras.csv", index=False)
print("Tabelas salvas em resultados/ e figuras em figuras/")

## Limitações
- **Correlação, não causa.** O estudo é transversal: não dá para afirmar que sentar mais causa os indicadores (nem o contrário).
- **Tempo sentado não é tempo de tela.** A pergunta soma TV, videogame, computador, celular, tablet e outras atividades sentado.
- **Agrupamento e Apriori não usam os pesos amostrais**, então descrevem os alunos da amostra; as porcentagens da seção 2 usam os pesos.
- **Exclusão do "não sei"** pode enviesar o tempo sentado se quem não sabe responder for diferente de quem sabe.
- **Separação fraca entre grupos:** a silhueta é baixa, comum em dados de questionário; os grupos descrevem um contínuo, não categorias naturais.
- Os indicadores de desesperança e vontade de se machucar tratam de temas sensíveis.